In [ ]:
import requests
import os
from dotenv import load_dotenv
import pprint

# 1. 環境變數設定
load_dotenv()
AUTHORIZATION = os.getenv("AUTHORIZATION")

#三十六小時天氣預報
url = 'https://opendata.cwa.gov.tw/api/v1/rest/datastore/F-C0032-001'
params = {
    'Authorization': AUTHORIZATION,
    'format': 'JSON'
}

response = requests.get(url, params=params)
data = response.json()
#print(data)

pprint.pprint(data)

In [ ]:
import requests
import os
from dotenv import load_dotenv

# 1. 環境變數設定
load_dotenv()
AUTHORIZATION = os.getenv("AUTHORIZATION")

# 2. 定義參數
# 注意：dataid 是路徑的一部分，不放入 params
# 氣象觀測站-全測站逐時氣象資料
dataid = 'O-A0001-001' 

# 3. 準備 Query Parameters (查詢參數)
# 這些參數會被 requests 自動轉換為 url?key=value 的形式
params ={
}

# 'StationId': 'C0D550',  # 雪霸為我們指定測站


# 4. 組合 URL 並發送請求
# 修正 Base URL 為 /api/v1/rest/datastore
base_url = "https://opendata.cwa.gov.tw/api/v1/rest/datastore"
url = f"{base_url}/{dataid}"

# 發送 GET 請求
res = requests.get(url, params=params)

# --- 觀察回應 ---
print(res.status_code)

data = res.json()

from pprint import pprint
pprint(data)

In [ ]:
import os
import requests
import pprint
from datetime import datetime, timedelta
from dotenv import load_dotenv

def fetch_cwa_sea_condition(station_id: str) -> dict:
    """
    獲取中央氣象署 48 小時內的海象監測資料。
    
    Args:
        station_id (str): 測站代碼 (例如龜山島為 "46708A")
        
    Returns:
        dict: 結構化的 JSON 資料。若請求失敗則回傳空字典。
    """
    # 1. 環境變數設定與驗證
    load_dotenv()
    authorization = os.getenv("AUTHORIZATION")
    if not authorization:
        raise ValueError("環境變數 AUTHORIZATION 未設定，請檢查 .env 檔案。")

    # 2. 定義 API 端點與路由
    dataid = 'O-B0075-001' 
    base_url = "https://opendata.cwa.gov.tw/api/v1/rest/datastore"
    url = f"{base_url}/{dataid}"

    # 3. 動態計算時間區段 (嚴格遵守 48 小時限制)
    now = datetime.now()
    past_48h = now - timedelta(hours=48)
    
    # 轉換為 ISO 8601 格式字串
    time_to_str = now.strftime('%Y-%m-%dT%H:%M:%S')
    time_from_str = past_48h.strftime('%Y-%m-%dT%H:%M:%S')

    # 4. 準備查詢參數
    params = {
        'Authorization': authorization,
        'StationID': station_id,
        'timeFrom': time_from_str,
        'timeTo': time_to_str
    }

    # 5. 發送 HTTP 請求與異常處理
    try:
        # 設定 timeout，避免伺服器無回應導致程式永久卡死
        response = requests.get(url, params=params, timeout=10)
        
        # 解析並回傳 JSON 資料
        return response.json()

# ==========================================
# 主程式執行區塊
# ==========================================
if __name__ == "__main__":
    # 龜山島測站
    target_station = "46708A"
    
    print(f"開始獲取測站 {target_station} 近 48 小時海象資料...")
    sea_data = fetch_cwa_sea_condition(station_id=target_station)
    
    if sea_data:
        print("資料獲取成功：")
        pprint.pprint(sea_data)
    else:
        print("未能獲取有效資料，請檢查日誌。")